UPscaledEV Project Main Code
Under the funding from TotalEnergies
Author: Yizhan Gu
Email: yig031@ucsd.edu
Affiliation: UCSD CER

Readme:
This code aims at postprocessing the raw session and interval EV data from PowerFlex into files that matches the optimization scripts.
# https://docs.google.com/presentation/d/1JRLf_Qb5xuxuLRH_28HUPq5HQBOIKCjCzQMGXw96MlE/edit?slide=id.g308df064212_0_68#slide=id.g308df064212_0_68

Labels:
NOTE: means there's a note and please read it
FIXME: means it's a bug or a problem that needs solving
TODO: means it's a to-do task, but not critical to the code running
VERSION: means there're more than one version for the diversity purpose, possibly shows in objective function choices or results analyses

Acknowledgement:
I gratefully acknowledge the support from my PI Jan Kleissl and TotalEnergies team, and the contributions from Yi-An Chen, whose prior work laid the foundation for this project. Special thanks to the UCSD Grid Lab team members.

Set working path and import packages

In [2]:
import os
os.chdir('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/2025Data/EV_data') 
print("Path is:", os.getcwd(), "\n")
import time
from datetime import datetime
import pandas as pd
from pathlib import Path 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

Path is: /Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/2025Data/EV_data 



1. Add battery size

In [ ]:
# Read raw session and interval EV data from PowerFlex

filename_Input = os.path.join("UCSD-All Sites - Portfolio sessions report 01_01_25-07_31_25.csv")
data_Sessions= pd.read_csv(filename_Input)
data_CarInfo= pd.read_csv("/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/PowerFlex input for EV statistics to be plugged into master sheet_2024.csv")

# Create extra columns of Car Year, Car Make, and Car Model for Session data
Vehicle_session = data_Sessions['Vehicle']
data_CarInfo['Year'] = data_CarInfo['Year'].astype(str)

Year = []
Make = []
Model = []
BESS = []

case = "new"

# If the Type of data is 'str', its the new data, if its 'int64', its the old data
if case == "new":
    
    for i in range(len(Vehicle_session)):
        #i = 4
        Vehicle_ThisSession = Vehicle_session[i]
        
        if pd.isna(Vehicle_ThisSession):
            Year_ThisSession = 'nan'
            Make_ThisSession = 'nan'
            Model_ThisSession = 'nan'
            BESS_ThisSession = 'nan'
        
        # the 'Vehicle' is NOT 0.000 in the PF session data
        elif len(Vehicle_ThisSession)>6:
            Year_ThisSession = Vehicle_ThisSession.split(' ')[0]
            Make_ThisSession = Vehicle_ThisSession.split(' ')[1]
            Model_ThisSession = Vehicle_ThisSession.split(Year_ThisSession + ' ' +Make_ThisSession+' ')[1]
            BESS_ThisSession = data_CarInfo['Battery (kWh)'][(data_CarInfo['Make']==Make_ThisSession)&\
                                                                (data_CarInfo['Model']==Model_ThisSession)&\
                                                                (data_CarInfo['Year']==Year_ThisSession) ]
           
            if len(pd.Series(BESS_ThisSession))>=1:
               BESS_ThisSession = BESS_ThisSession.iloc[0] 
                
            #if no match, give up one matching the Year, as long as the Make and Model match, asign the BES capacity
            elif len(pd.Series(BESS_ThisSession))==0:
                BESS_ThisSession = data_CarInfo['Battery (kWh)'][(data_CarInfo['Make']==Make_ThisSession)&\
                                                                    (data_CarInfo['Model']==Model_ThisSession)]
            
                if len(pd.Series(BESS_ThisSession))>=1:
                   BESS_ThisSession = BESS_ThisSession.iloc[0]
                   
                #if STILL no match, the BESS capacity is unknown
                elif len(pd.Series(BESS_ThisSession))==0:
                    BESS_ThisSession = 'nan'

       
        
        # if the 'Vehicle' is 0.000 in the PF session data
        else:
            Year_ThisSession = 'nan'
            Make_ThisSession = 'nan'
            Model_ThisSession = 'nan'
            BESS_ThisSession = 'nan'
            
            
                    
        Year = Year + [Year_ThisSession]
        Make = Make + [Make_ThisSession]
        Model = Model + [Model_ThisSession]
        BESS = BESS + [BESS_ThisSession]            
            
            

        

else:
    
    for i in range(len(Vehicle_session)):
        Vehicle_ThisSession = Vehicle_session[i]
        data_CarInfo_ThisSession = data_CarInfo[data_CarInfo['Doe Id']==Vehicle_ThisSession]

        if len(data_CarInfo_ThisSession)==0:
            Year = Year + ['nan']
            Make = Make + ['nan']
            Model = Model + ['nan']
            BESS = BESS + ['nan']
            
        else:
            Year = Year + [data_CarInfo_ThisSession['Year'].iloc[0]]
            Make = Make + [data_CarInfo_ThisSession['Make'].iloc[0]]
            Model = Model + [data_CarInfo_ThisSession['Model'].iloc[0]]
            BESS = BESS + [data_CarInfo_ThisSession['Battery (kWh)'].iloc[0]]




data_Sessions['Car Year'] = Year
data_Sessions['Car Make'] = Make
data_Sessions['Car Model'] = Model
data_Sessions['Car Battery'] = BESS


# Write the Session_wCarInfo data to .csv file
data_Sessions.to_csv('UCSD_AllSites_Sessions_wCarInfo.csv', index=False)



2. Merge session data with raw interval data

In [ ]:

TheDate_Start = datetime(2025, 1, 1)
TheDate_End = datetime(2025, 7, 31)

TheDate_Start_str = TheDate_Start.strftime("%Y%m%d")
TheDate_End_str = TheDate_End.strftime("%Y%m%d")

data_Interval= pd.read_csv("UCSD-All Sites - Portfolio interval report 01_01_25-07_31_25.csv")

# Filter the data based on the duration you choose

IntervalStart_datetime = pd.to_datetime(data_Interval['Interval start'])
IntervalEnd_datetime = pd.to_datetime(data_Interval['Interval end'])

SessionStart_datetime = pd.to_datetime(data_Sessions['Session start'])
SessionEnd_datetime = pd.to_datetime(data_Sessions['Session end'])

data_Interval_MySite_MyDates = data_Interval[(TheDate_Start <= IntervalStart_datetime) & (IntervalStart_datetime <= TheDate_End) \
                                                & (TheDate_Start <= IntervalEnd_datetime) & (IntervalEnd_datetime <= TheDate_End) ]

data_Sessions_MySite_MyDates = data_Sessions[(TheDate_Start <= SessionStart_datetime) & (SessionStart_datetime <= TheDate_End) \
                                                & (TheDate_Start <= SessionEnd_datetime) & (SessionEnd_datetime <= TheDate_End) ]


# Re asign index to the filtered Interval/Session data

data_Interval_MySite_MyDates_NewInd = data_Interval_MySite_MyDates.copy()
data_Sessions_MySite_MyDates_NewInd = data_Sessions_MySite_MyDates.copy()

# rename column 10-digit session UID in data_sessions:
data_Sessions_MySite_MyDates_NewInd = data_Sessions_MySite_MyDates_NewInd.rename(columns={'10-digit session UID': '10-digit UID'})

UID_list_Interval = data_Interval_MySite_MyDates_NewInd["10-digit UID"].unique()
UID_list_Sessions = data_Sessions_MySite_MyDates_NewInd["10-digit UID"].unique()

ColumnNames_appen = [\
    "10-digit UID", \
    "Session start", "Session end", "Session duration (minutes)", "Charging duration (minutes)","Session idle (minutes)", "kWh delivered",\
    "SoC Start","SoC End","User", "Vehicle", "EVSE Status", \
    "Serial #", \
    "Parking Space", "Site", \
    "Cost to site", "Cost to driver", "Fleet", "Vehicle barcode", "Minutes available", "Miles needed", "Energy needed", "Energy unit", "Wh per mile",\
    'Car Year', 'Car Make', 'Car Model', 'Car Battery']    
    
data_Sessions_appen = data_Sessions_MySite_MyDates_NewInd.filter(ColumnNames_appen)

# Merge the Sessions and the Interval data by "10-digit UID"
data_Interval_MySite_aux = []
UID_missing = list()

import pandas as pd

data_Interval_MySite_aux = pd.merge(
    data_Interval_MySite_MyDates_NewInd,
    data_Sessions_appen,
    on="10-digit UID",
    how="inner"
)

# UIDs that are missing from sessions
UID_missing = set(UID_list_Interval) - set(data_Sessions_appen["10-digit UID"])

print("Missing UIDs ratio: ", len(UID_missing)/len(UID_list_Interval) * 100, "%")

data_Interval_MySite_aux.to_csv("UCSD_AllSites_Merge.csv", index=False)


3. Postprocessing interval data

In [ ]:
data_Merge = pd.read_csv("UCSD_AllSites_Merge.csv", low_memory=False)
data_Merge_flip = data_Merge.iloc[::-1]


data_Merge_NoDataAvailable = data_Merge_flip[data_Merge_flip["Interval kWh"]=="-"]
Events_NoDataAvailable = data_Merge_NoDataAvailable["10-digit UID"].unique()
data_Merge_NoDataAvailable_ = data_Merge_flip[data_Merge_flip["10-digit UID"].isin(Events_NoDataAvailable)]

# Case 1: For column "Interval kWh", a series of "No data available" is followed by a positive value

data_Merge_NoDataAvailable_new = pd.DataFrame()
numb_FollowedByPos = 0
Event_FollowedByPos = [] 


for i in tqdm(range(len(Events_NoDataAvailable)), desc="Processing Events"):
    ThisEvent = Events_NoDataAvailable[i]
    data_Merge_NoDataAvailable_ThisEvent = data_Merge_NoDataAvailable_[data_Merge_NoDataAvailable_["10-digit UID"] == ThisEvent]
    numb_row_ThisEvent = data_Merge_NoDataAvailable_ThisEvent.shape[0]  # Gives number of rows
    count = 0
    for j in range(numb_row_ThisEvent - 1):
        if data_Merge_NoDataAvailable_ThisEvent["Interval kWh"].iloc[j] == "-":
            if data_Merge_NoDataAvailable_ThisEvent["Interval kWh"].iloc[j+1] == "-":
                count = count + 1
            elif float(data_Merge_NoDataAvailable_ThisEvent["Interval kWh"].iloc[j+1]) >= 0:
                numb_FollowedByPos += 1
                Event_FollowedByPos.append(ThisEvent)
                numb_NoDataAvailable = count + 2
                numb_Accumulated_kWh = float(data_Merge_NoDataAvailable_ThisEvent["Interval kWh"].iloc[j+1])
                data_Merge_NoDataAvailable_ThisEvent.loc[j-count:j, "Interval kWh"] = numb_Accumulated_kWh / numb_NoDataAvailable
                count_final = count
                count = 0

    data_Merge_NoDataAvailable_new = pd.concat(
        [data_Merge_NoDataAvailable_new, data_Merge_NoDataAvailable_ThisEvent],
        ignore_index=True
    )
    
# Case 2: For column "Interval kWh", a series of "No data available" is followed by a non positive value
data_Merge_NoDataAvailable_new_ = data_Merge_NoDataAvailable_new[data_Merge_NoDataAvailable_new["Interval kWh"]=="-"]
Event_FollowedByNoData = data_Merge_NoDataAvailable_new_['10-digit UID'].unique()


data_Merge_NoDataAvailable_new["Interval kWh"][data_Merge_NoDataAvailable_new["Interval kWh"]=="-"] = 0
data_Merge_NoDataAvailable_new["Interval kWh"] = data_Merge_NoDataAvailable_new["Interval kWh"].astype(float)


Event_FollowedByNeg = data_Merge_NoDataAvailable_new['10-digit UID'][data_Merge_NoDataAvailable_new["Interval kWh"] <0]
Event_FollowedByNeg_unique = np.unique(np.array(Event_FollowedByNeg))
data_Merge_NoDataAvailable_new["Interval kWh"][data_Merge_NoDataAvailable_new["Interval kWh"]<0] = 0


# Case 3: Combine the processed interval data with "No data available" with the rest interval data

data_Merge_Rest = data_Merge_flip[~data_Merge_flip["10-digit UID"].isin(Events_NoDataAvailable)]

data_Merged_Processed = pd.concat(
    [data_Merge_Rest, data_Merge_NoDataAvailable_new],
    ignore_index=True
)
data_Merged_Processed = data_Merged_Processed[data_Merged_Processed['Interval kWh']!='-']
data_Merged_Processed["Interval kWh"] = data_Merged_Processed["Interval kWh"].to_numpy(dtype=float)


Event_Processed_below0_unique = np.unique(np.array(data_Merged_Processed['10-digit UID'][data_Merged_Processed["Interval kWh"] <0]))
data_Merged_Processed["Interval kWh"] = data_Merged_Processed["Interval kWh"].clip(lower=0)

Event_Processed_above3_unique = np.unique(np.array(data_Merged_Processed['10-digit UID'][data_Merged_Processed["Interval kWh"] > 4.16]))

data_Merged_Processed["Interval kWh"] = data_Merged_Processed["Interval kWh"].clip(upper=4.16)


data_Merged_Processed.to_csv("UCSD_AllSites_Merge_PostProcessedInterval.csv", index=False)


AbnormalSessionStart_list = []
data_Merge_abnormal_list = []    
Event_DontMatch = []
Events = data_Merged_Processed["10-digit UID"].unique() 

for ind_event in tqdm(range(len(Events)), desc="Processing Events"):
    uid = Events[ind_event]
    mask = data_Merged_Processed["10-digit UID"] == uid
    data_Merge_2_ThisEvent = data_Merged_Processed.loc[mask]

    if data_Merge_2_ThisEvent["Interval kWh"].sum() - data_Merge_2_ThisEvent["kWh delivered"].iloc[0] != 0:
        Event_DontMatch.append(uid)
        data_Merged_Processed.loc[mask, "kWh delivered"] = data_Merge_2_ThisEvent["Interval kWh"].sum()
        
        


# Create a new column "Eta" and assign Eta values to each UID
unique_uids = data_Merged_Processed["10-digit UID"].unique()

# Randomly assign Eta values equally among [0.2, 0.4, 0.6, 0.8, 1] for each UID
eta_choices = np.array([0.2, 0.4, 0.6, 0.8, 1])
num_uids = len(unique_uids)
repeats = num_uids // len(eta_choices)
remainder = num_uids % len(eta_choices)
eta_values = np.tile(eta_choices, repeats)
if remainder > 0:
    eta_values = np.concatenate([eta_values, np.random.choice(eta_choices, remainder, replace=False)])
np.random.shuffle(eta_values)

# Create a mapping from UID to Eta
uid_eta_map = dict(zip(unique_uids, eta_values))

# Map Eta to each row in the dataframe
data_Merged_Processed["Eta"] = data_Merged_Processed["10-digit UID"].map(uid_eta_map)

data_Merged_Processed.to_csv('UCSD_AllSites_Merge_PostProcessedSession.csv', index=False)




Processing Events: 100%|██████████| 6234/6234 [01:22<00:00, 75.13it/s] 
